In [2]:
import pandas as pd
import numpy as np
import os


In [3]:
crime_data = pd.read_csv("yorkshire_crime_2024.csv")
perception_df = pd.read_csv("Perceptions_CJS_England and Wales_2024Q4.csv")
ref_df = pd.read_csv("reference-measurementvar.csv")

In [4]:
crime_data.head()

,force_name,month,category
0,South Yorkshire,2024-01,bicycle-theft
1,South Yorkshire,2024-01,bicycle-theft
2,South Yorkshire,2024-01,bicycle-theft
3,South Yorkshire,2024-01,bicycle-theft
4,South Yorkshire,2024-01,burglary


In [5]:
perception_df.head()


,Year,Quarter,MeasurementVar,MeasurementLevel,MeasurementType,Geography,HouseholdType,Sex,Age,CharacteristicVar,Characteristic,Estimate,StandardError,UnweightedCount
0,2024,4,catt2,Person,Perception,England and Wales,,All people,16-24,accharm3,Detached house,49.81129436,3.989754607,229
1,2024,4,catt2,Person,Perception,England and Wales,,All people,16-24,accharm3,Flats/maisonettes,45.58845912,3.317115821,460
2,2024,4,catt2,Person,Perception,England and Wales,,All people,16-24,accharm3,House,51.5229105,1.880560431,1194
3,2024,4,catt2,Person,Perception,England and Wales,,All people,16-24,accharm3,Other accommodation,[x],[x],[x]
4,2024,4,catt2,Person,Perception,England and Wales,,All people,16-24,accharm3,Semi-detached house,52.39973521,3.208098498,418


In [6]:
ref_df.head()

,MeasurementVar,MeasurementLabel,MeasurementLevel,MeasurementType
0,abancar1,Perceived problem with abandoned or burnt out ...,Person,Perception
1,acquai_i,Acquaintance violence,Person,Incidence rate
2,acquai_p,Acquaintance violence,Person,Prevalence rate
3,afee_i,Advance fee fraud,Person,Incidence rate
4,afee_p,Advance fee fraud,Person,Prevalence rate


In [7]:
crime_categories = crime_data['category'].unique()

In [8]:
crime_categories

array(['bicycle-theft', 'burglary', 'criminal-damage-arson', 'drugs',
       'other-theft', 'possession-of-weapons', 'public-order', 'robbery',
       'shoplifting', 'theft-from-the-person', 'vehicle-crime',
       'violent-crime', 'other-crime', 'anti-social-behaviour'],
      dtype=object)

In [9]:
ref_crime_cat = ref_df["MeasurementLabel"].unique()
ref_crime_cat

array(['Perceived problem with abandoned or burnt out cars in the local area',
       'Acquaintance violence', 'Advance fee fraud',
       'Domestic burglary with entry and no loss',
       'Domestic burglary with entry and loss',
       'Attempted domestic burglary', 'Domestic burglary with entry',
       'Domestic burglary', 'All vehicle crime',
       'All vehicle-related theft', 'All vehicle-related thefts',
       'All theft', 'Attempted theft of and from vehicles',
       'Bank and credit account fraud', 'Bicycle theft',
       'Burglary in building other than dwelling with entry and no loss',
       'Burglary in building other than dwelling with entry and loss',
       'Burglary in dwelling with no loss',
       'Attempted burglary in dwelling',
       'Burglary in dwelling with entry', 'Burglary in dwelling',
       'Burglary in dwelling with loss',
       'Burglary in building other than dwelling',
       'Attempted burglary in building other than dwelling',
       'Burglary i

In [10]:
# Define the crime categories
crime_categories = np.array(['bicycle-theft', 'burglary', 'criminal-damage-arson', 'drugs',
       'other-theft', 'possession-of-weapons', 'public-order', 'robbery',
       'shoplifting', 'theft-from-the-person', 'vehicle-crime',
       'violent-crime', 'other-crime', 'anti-social-behaviour'])

# Load the CSV file
df = pd.read_csv("reference-measurementvar.csv")

# Function to categorize based on the MeasurementLabel
def categorize_crime(label):
    label = label.lower()
    
    # Bicycle theft
    if "bicycle theft" in label:
        return "bicycle-theft"
    
    # Burglary
    elif any(x in label for x in ["burglary", "burgla"]):
        return "burglary"
    
    # Criminal damage and arson
    elif any(x in label for x in ["criminal damage", "arson", "vandal"]):
        return "criminal-damage-arson"
    
    # Drugs
    elif "drug" in label:
        return "drugs"
    
    # Vehicle crime
    elif any(x in label for x in ["vehicle", "car crime", "motor vehicle"]):
        return "vehicle-crime"
    
    # Robbery
    elif "robbery" in label or "mugging" in label:
        return "robbery"
    
    # Theft from the person
    elif "theft from the person" in label or "snatch theft" in label or "stealth theft" in label:
        return "theft-from-the-person"
    
    # Violent crime
    elif any(x in label for x in ["violence", "violent", "assault", "wounding", "injury"]):
        return "violent-crime"
    
    # Other theft
    elif any(x in label for x in ["theft", "thft"]) and not any(x in label for x in ["vehicle", "bicycle", "person"]):
        return "other-theft"
    
    # Public order
    elif "drunk" in label or "rowdy" in label:
        return "public-order"
    
    # Anti-social behaviour
    elif "anti-social behaviour" in label or "teenagers hanging around" in label or "noisy neighbours" in label:
        return "anti-social-behaviour"
    
    # Fraud and computer misuse
    elif any(x in label for x in ["fraud", "computer misuse", "hacking", "virus"]):
        return "other-crime"
    
    # Return NaN for perception measures about police performance, CJS opinions, etc.
    elif any(x in label for x in ["police", "can be relied", "treat", "respect", "confidence", "dealing with", "trusted", 
                              "council", "matters in this area", "CJS", "fair", "effectiveness", 
                              "perceived change", "crime level", "worried about"]):
        return np.nan
    
    # For other actual crime types and anti-social behavior
    else:
        if "hanging around" in label or "drunk" in label or "rowdy" in label or "noisy" in label:
            return "anti-social-behaviour"
        elif any(x in label for x in ["abandoned", "burnt out cars", "litter", "rubbish", "problem with"]):
            return "anti-social-behaviour"
        elif "all" in label and any(x in label for x in ["crime", "household", "personal"]):
            return "other-crime"
        return "other-crime"

# Apply the categorization function to create a new column
df['CrimeCategory'] = df['MeasurementLabel'].apply(categorize_crime)

# Save the updated dataframe to a new CSV file
df.to_csv("reference-measurementvar-with-categories.csv", index=False)

# Display summary of categories
category_counts = df['CrimeCategory'].value_counts(dropna=False)
print("Crime Category Distribution:")
print(category_counts)

# Calculate percentage of NaN values
nan_percentage = (df['CrimeCategory'].isna().sum() / len(df)) * 100
print(f"\nPercentage of entries with NaN category: {nan_percentage:.2f}%")

# Display a sample of the updated dataframe
print("\nSample of updated dataframe:")
print(df[['MeasurementVar', 'MeasurementLabel', 'CrimeCategory']].head(10))

# Show some examples of records with NaN categories
print("\nExamples of records with NaN categories:")
nan_examples = df[df['CrimeCategory'].isna()][['MeasurementVar', 'MeasurementLabel', 'CrimeCategory']].head(5)
print(nan_examples)

Crime Category Distribution:
CrimeCategory
other-crime              31
burglary                 31
violent-crime            21
NaN                      15
vehicle-crime            11
other-theft               9
theft-from-the-person     8
criminal-damage-arson     7
anti-social-behaviour     5
robbery                   4
bicycle-theft             2
drugs                     1
public-order              1
Name: count, dtype: int64

Percentage of entries with NaN category: 10.27%

Sample of updated dataframe:
  MeasurementVar                                   MeasurementLabel  \
0       abancar1  Perceived problem with abandoned or burnt out ...   
1       acquai_i                              Acquaintance violence   
2       acquai_p                              Acquaintance violence   
3         afee_i                                  Advance fee fraud   
4         afee_p                                  Advance fee fraud   
5       albenl_i           Domestic burglary with entry and no

In [13]:
# Ensure that the cell defining 'df' (CELL INDEX: 8) is executed before running this cell.

# Remove trailing spaces and convert to lowercase in df MeasurementVar
df['MeasurementVar'] = df['MeasurementVar'].str.strip().str.lower()

In [14]:
perception_df['MeasurementVar'] = perception_df['MeasurementVar'].str.strip().str.lower()
# Merge the two dataframes on 'MeasurementVar'

In [16]:
# check to see measurement var in both dataframes match
df['MeasurementVar'].unique()


array(['abancar1', 'acquai_i', 'acquai_p', 'afee_i', 'afee_p', 'albenl_i',
       'albenl_p', 'albuel_i', 'albuel_p', 'albura_i', 'albura_p',
       'albure_i', 'albure_p', 'alburg_i', 'alburg_p', 'allmvc_i',
       'allmvc_p', 'allmvt_i', 'allmvt_p', 'althft_p', 'attmvt_i',
       'attmvt_p', 'bank_i', 'bank_p', 'biketh_i', 'biketh_p', 'bndenl_i',
       'bndenl_p', 'brndel_i', 'brndel_p', 'burdnl_i', 'burdnl_p',
       'burgat_i', 'burgat_p', 'burgen_i', 'burgen_p', 'burgla_i',
       'burgla_p', 'burglo_i', 'burglo_p', 'burgnd_i', 'burgnd_p',
       'burnda_i', 'burnda_p', 'burnde_i', 'burnde_p', 'bvpiburg',
       'bvpicar', 'bvpifrd', 'bvpiviol', 'catt2', 'catt2a', 'catt2b',
       'cjsovb1dv', 'clcntry2', 'clloc2', 'com_i', 'com_p', 'cominj_i',
       'cominj_p', 'common_i', 'common_p', 'comnij_i', 'comnij_p',
       'domest_i', 'domest_p', 'drug1', 'drunk1', 'fairova1dv', 'frd_i',
       'frd_p', 'frdcom_i', 'frdcom_p', 'hackua_i', 'hackua_p',
       'homeva_i', 'homeva_p', 'mug

In [17]:
perception_df['MeasurementVar'].unique()


array(['catt2', 'catt2a', 'catt2b', 'cjsovb1dv', 'fairova1dv', 'patt1',
       'patt2', 'patt3', 'patt5', 'patt6', 'patt6b', 'patt7', 'ratpol3'],
      dtype=object)